<a href="https://colab.research.google.com/github/pnperl/Travel_Router/blob/main/Travel_Router.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ============================================================
# SMART INDIAN TRAVEL ROUTE ENGINE - PHASE 1
# ============================================================

!pip -q install pandas tabulate ipywidgets

import pandas as pd
from tabulate import tabulate
from datetime import datetime, timedelta
import ipywidgets as widgets
from IPython.display import display

# ============================================================
# CITY → STATION MAPPING
# ============================================================

CITY_TO_STATIONS = {

    "JAMNAGAR": {
        "rail": ["JAM"],
        "bus": ["Jamnagar Bus Stand"]
    },

    "RAJKOT": {
        "rail": ["RJT"],
        "bus": ["Rajkot Bus Stand"]
    },

    "AHMEDABAD": {
        "rail": ["ADI", "SBIB"],
        "bus": ["Gita Mandir"]
    },

    "SURAT": {
        "rail": ["ST"],
        "bus": ["Surat Central"]
    },

    "VADODARA": {
        "rail": ["BRC"],
        "bus": ["Vadodara Central"]
    },

    "MUMBAI": {
        "rail": [
            "MMCT",
            "BDTS",
            "DDR",
            "LTT",
            "CSMT",
            "PNVL"
        ],

        "bus": [
            "Borivali",
            "Dadar",
            "Mumbai Central"
        ]
    }
}

# ============================================================
# UI
# ============================================================

cities = sorted(CITY_TO_STATIONS.keys())

source_widget = widgets.Combobox(
    placeholder='Type source city',
    options=cities,
    description='From:',
    ensure_option=True,
    layout=widgets.Layout(width='400px')
)

dest_widget = widgets.Combobox(
    placeholder='Type destination city',
    options=cities,
    description='To:',
    ensure_option=True,
    layout=widgets.Layout(width='400px')
)

group_widget = widgets.IntSlider(
    value=10,
    min=1,
    max=30,
    step=1,
    description='Passengers:',
    layout=widgets.Layout(width='400px')
)

boarding_widget = widgets.Checkbox(
    value=True,
    description='Allow nearby boarding stations'
)

deboarding_widget = widgets.Checkbox(
    value=True,
    description='Allow nearby destination stations'
)

print("SELECT JOURNEY DETAILS\n")

display(source_widget)
display(dest_widget)
display(group_widget)
display(boarding_widget)
display(deboarding_widget)

# ============================================================
# SAMPLE TRAIN DATABASE
# ============================================================

TRAINS = [

    {
        "train": "Saurashtra Mail",
        "from": "JAM",
        "to": "DDR",
        "dep": "20:30",
        "arr": "10:30",
        "class": "3A",
        "availability": 8
    },

    {
        "train": "Saurashtra Janta",
        "from": "JAM",
        "to": "BDTS",
        "dep": "21:45",
        "arr": "12:00",
        "class": "SL",
        "availability": 72
    },

    {
        "train": "Humsafar Express",
        "from": "JAM",
        "to": "BDTS",
        "dep": "19:30",
        "arr": "09:30",
        "class": "3A",
        "availability": 12
    },

    {
        "train": "Saurashtra Express",
        "from": "JAM",
        "to": "ADI",
        "dep": "13:00",
        "arr": "19:00",
        "class": "CC",
        "availability": 45
    },

    {
        "train": "Saurashtra Mail",
        "from": "JAM",
        "to": "ADI",
        "dep": "20:00",
        "arr": "04:30",
        "class": "SL",
        "availability": 58
    },

    {
        "train": "Intercity",
        "from": "JAM",
        "to": "RJT",
        "dep": "16:00",
        "arr": "18:30",
        "class": "2S",
        "availability": 120
    },

    {
        "train": "Gujarat Mail",
        "from": "ADI",
        "to": "MMCT",
        "dep": "22:00",
        "arr": "06:00",
        "class": "3A",
        "availability": 28
    },

    {
        "train": "Garib Rath",
        "from": "ADI",
        "to": "BDTS",
        "dep": "23:00",
        "arr": "08:00",
        "class": "3A",
        "availability": 36
    },

    {
        "train": "Lok Shakti",
        "from": "ADI",
        "to": "BDTS",
        "dep": "20:30",
        "arr": "07:00",
        "class": "3A",
        "availability": 14
    },

    {
        "train": "Saurashtra Mail",
        "from": "RJT",
        "to": "DDR",
        "dep": "21:00",
        "arr": "10:30",
        "class": "3A",
        "availability": 9
    },

    {
        "train": "Saurashtra Janta",
        "from": "RJT",
        "to": "BDTS",
        "dep": "22:30",
        "arr": "12:00",
        "class": "SL",
        "availability": 62
    }
]

df = pd.DataFrame(TRAINS)

# ============================================================
# HELPERS
# ============================================================

def get_city_stations(city):

    city = city.upper()

    if city not in CITY_TO_STATIONS:
        return []

    return CITY_TO_STATIONS[city]["rail"]

def parse_time(t):

    return datetime.strptime(t, "%H:%M")

def layover_minutes(arr, dep):

    arr_dt = parse_time(arr)
    dep_dt = parse_time(dep)

    if dep_dt < arr_dt:
        dep_dt += timedelta(days=1)

    return int((dep_dt - arr_dt).seconds / 60)

def is_night(dep):

    hr = int(dep.split(":")[0])

    return hr >= 19 or hr <= 6

# ============================================================
# ROUTE SEARCH
# ============================================================

def find_routes():

    SOURCE_CITY = source_widget.value.upper()
    DEST_CITY = dest_widget.value.upper()

    GROUP_SIZE = group_widget.value

    if SOURCE_CITY not in CITY_TO_STATIONS:
        print("Invalid source city")
        return

    if DEST_CITY not in CITY_TO_STATIONS:
        print("Invalid destination city")
        return

    if boarding_widget.value:
        source_stations = get_city_stations(SOURCE_CITY)
    else:
        source_stations = [get_city_stations(SOURCE_CITY)[0]]

    if deboarding_widget.value:
        dest_stations = get_city_stations(DEST_CITY)
    else:
        dest_stations = [get_city_stations(DEST_CITY)[0]]

    results = []

    # ========================================================
    # DIRECT ROUTES
    # ========================================================

    direct = df[
        (df["from"].isin(source_stations)) &
        (df["to"].isin(dest_stations))
    ]

    for _, row in direct.iterrows():

        score = 0

        score += min(row["availability"], 50)

        if row["availability"] >= GROUP_SIZE:
            score += 100

        if row["class"] == "3A":
            score += 40

        if is_night(row["dep"]):
            score += 20

        results.append({

            "type": "DIRECT",

            "route":
                f"{row['from']} -> {row['to']}",

            "journey":
                f"{row['train']} "
                f"({row['dep']} - {row['arr']}) "
                f"{row['class']}",

            "availability":
                row["availability"],

            "recommended":
                "YES"
                if row["availability"] >= GROUP_SIZE
                else "PARTIAL",

            "score":
                score
        })

    # ========================================================
    # HOPPING ROUTES
    # ========================================================

    first_legs = df[df["from"].isin(source_stations)]

    for _, leg1 in first_legs.iterrows():

        hub = leg1["to"]

        second_legs = df[
            (df["from"] == hub) &
            (df["to"].isin(dest_stations))
        ]

        for _, leg2 in second_legs.iterrows():

            layover = layover_minutes(
                leg1["arr"],
                leg2["dep"]
            )

            if layover < 45:
                continue

            min_avail = min(
                leg1["availability"],
                leg2["availability"]
            )

            score = 0

            score += min(min_avail, 50)

            if min_avail >= GROUP_SIZE:
                score += 100

            if leg2["class"] == "3A":
                score += 40

            if is_night(leg2["dep"]):
                score += 20

            results.append({

                "type": "HOPPING",

                "route":
                    f"{leg1['from']} -> "
                    f"{hub} -> "
                    f"{leg2['to']}",

                "journey":
                    f"{leg1['train']} "
                    f"({leg1['dep']}-{leg1['arr']}) "
                    f"{leg1['class']}\n"
                    f"↓ Layover {layover} mins ↓\n"
                    f"{leg2['train']} "
                    f"({leg2['dep']}-{leg2['arr']}) "
                    f"{leg2['class']}",

                "availability":
                    min_avail,

                "recommended":
                    "YES"
                    if min_avail >= GROUP_SIZE
                    else "PARTIAL",

                "score":
                    score
            })

    results_df = pd.DataFrame(results)

    if len(results_df) == 0:
        print("\nNo routes found")
        return

    results_df = results_df.sort_values(
        by="score",
        ascending=False
    )

    print("\n")
    print("=" * 100)
    print("BEST ROUTE OPTIONS")
    print("=" * 100)
    print("\n")

    print(tabulate(
        results_df[
            [
                "type",
                "route",
                "journey",
                "availability",
                "recommended",
                "score"
            ]
        ],
        headers="keys",
        tablefmt="fancy_grid",
        showindex=False
    ))

# ============================================================
# RUN BUTTON
# ============================================================

run_button = widgets.Button(
    description="Find Best Routes",
    button_style='success',
    layout=widgets.Layout(width='250px')
)

display(run_button)

run_button.on_click(lambda x: find_routes())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 47.6 MB/s eta 0:00:00
SELECT JOURNEY DETAILS



Combobox(value='', description='From:', ensure_option=True, layout=Layout(width='400px'), options=('AHMEDABAD'…

Combobox(value='', description='To:', ensure_option=True, layout=Layout(width='400px'), options=('AHMEDABAD', …

IntSlider(value=10, description='Passengers:', layout=Layout(width='400px'), max=30, min=1)

Checkbox(value=True, description='Allow nearby boarding stations')

Checkbox(value=True, description='Allow nearby destination stations')

Button(button_style='success', description='Find Best Routes', layout=Layout(width='250px'), style=ButtonStyle…



BEST ROUTE OPTIONS


╒════════╤═════════════╤═════════════════════════════════╤════════════════╤═══════════════╤═════════╕
│ type   │ route       │ journey                         │   availability │ recommended   │   score │
╞════════╪═════════════╪═════════════════════════════════╪════════════════╪═══════════════╪═════════╡
│ DIRECT │ ADI -> BDTS │ Garib Rath (23:00 - 08:00) 3A   │             36 │ YES           │     196 │
├────────┼─────────────┼─────────────────────────────────┼────────────────┼───────────────┼─────────┤
│ DIRECT │ ADI -> MMCT │ Gujarat Mail (22:00 - 06:00) 3A │             28 │ YES           │     188 │
├────────┼─────────────┼─────────────────────────────────┼────────────────┼───────────────┼─────────┤
│ DIRECT │ ADI -> BDTS │ Lok Shakti (20:30 - 07:00) 3A   │             14 │ YES           │     174 │
╘════════╧═════════════╧═════════════════════════════════╧════════════════╧═══════════════╧═════════╛
